# Data.org Financial Health Prediction Challenge!

## Empowering Financial Inclusion Across Africa

### Unzip Dataset

In [4]:
!ls

 LICENSE
 dataorg-financial-health-prediction-challenge.zip
'dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n (1).zip:Zone.Identifier'
 financial-health-zindi-gbms.ipynb


In [5]:
!unzip dataorg-financial-health-prediction-challenge.zip

Archive:  dataorg-financial-health-prediction-challenge.zip
  inflating: manifest-c388a32489998f91d25543edfbe8b78f20251204-19827-1nvty2o.json  
  inflating: Test.csv                
  inflating: Train.csv               
  inflating: SampleSubmission.csv    
  inflating: VariableDefinitions.csv  
  inflating: Starter Notebook.ipynb  


## Library Import

In [72]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from numpy import random
from tqdm import tqdm
from pathlib import Path

from fastai.tabular.all import *
from ipywidgets import interact

from fastai.imports import *
np.set_printoptions(linewidth=130)
import gc

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import f1_score

import xgboost as xgb
from xgboost import plot_importance

import lightgbm as lgb
import catboost as cat

In [69]:
# GPU Availability Check for XGBoost
print("="*70)
print("GPU AVAILABILITY CHECK")
print("="*70)

import subprocess
import sys

# Check if CUDA is available
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print("✅ NVIDIA GPU detected!")
        print("\nGPU Information:")
        print("-" * 70)
        # Parse nvidia-smi output for GPU name and memory
        for line in result.stdout.split('\n'):
            if 'NVIDIA' in line or 'CUDA' in line:
                print(line.strip())
        print("-" * 70)
        
        # Check XGBoost GPU support
        try:
            import xgboost as xgb
            # Try to create a simple GPU-based model
            test_params = {'device': 'cuda', 'tree_method': 'hist'}
            test_model = xgb.XGBClassifier(**test_params)
            print("\n✅ XGBoost GPU support is AVAILABLE!")
            print("   You can use: device='cuda', tree_method='hist'")
            gpu_available = True
        except Exception as e:
            print(f"\n⚠️  XGBoost installed but GPU support not working: {e}")
            print("   You may need to reinstall XGBoost with GPU support:")
            print("   pip install xgboost --upgrade")
            gpu_available = False
    else:
        print("❌ No NVIDIA GPU detected")
        gpu_available = False
except FileNotFoundError:
    print("❌ nvidia-smi not found - No NVIDIA GPU available")
    print("   Running on CPU only")
    gpu_available = False
except subprocess.TimeoutExpired:
    print("⚠️  nvidia-smi timeout - GPU may not be properly configured")
    gpu_available = False
except Exception as e:
    print(f"⚠️  Error checking GPU: {e}")
    gpu_available = False

# Set device based on availability
if gpu_available:
    DEVICE = 'cuda'
    print(f"\n🚀 Device set to: {DEVICE} (GPU acceleration enabled)")
else:
    DEVICE = 'cpu'
    print(f"\n💻 Device set to: {DEVICE} (CPU only)")

print("="*70)
print(f"\nUse this in your XGBoost parameters: 'device': '{DEVICE}'")
print("="*70)

GPU AVAILABILITY CHECK
❌ nvidia-smi not found - No NVIDIA GPU available
   Running on CPU only

💻 Device set to: cpu (CPU only)

Use this in your XGBoost parameters: 'device': 'cpu'


## Load Dataset

In [2]:
path = Path('')
path

Path('.')

In [3]:
!ls

 LICENSE
 SampleSubmission.csv
'Starter Notebook.ipynb'
 Test.csv
 Train.csv
 VariableDefinitions.csv
 dataorg-financial-health-prediction-challenge.zip
'dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n (1).zip:Zone.Identifier'
 financial-health-zindi-gbms.ipynb
 manifest-c388a32489998f91d25543edfbe8b78f20251204-19827-1nvty2o.json
 submission_fastai.csv


In [4]:
train_df = pd.read_csv(path/'Train.csv',index_col='ID')
test_df = pd.read_csv(path/'Test.csv',index_col='ID')
sub_df = pd.read_csv(path/'SampleSubmission.csv')
vd_df = pd.read_csv(path/'VariableDefinitions.csv')

## EDA

In [5]:
train_df.head()

,country,owner_age,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,personal_income,business_expenses,business_turnover,...,has_internet_banking,has_debit_card,future_risk_theft_stock,business_age_months,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,Target
ID,,,,,,,,,,,,,,,,,,,,,
ID_3CFL0U,eswatini,63.0,Yes,No,No,No,Yes,3000.0,6000.0,7000.0,...,Never had,Never had,NaN,6.0,Never had,Used to have but don’t have now,NaN,Never had,Never had,Low
ID_XWI7G3,zimbabwe,39.0,No,Yes,Yes,No,Yes,NaN,NaN,NaN,...,NaN,NaN,No,3.0,Never had,Never had,NaN,NaN,NaN,Medium
ID_TY93LV,malawi,34.0,Don’t know or N/A,No,No,Don't know,Yes,30000.0,6000.0,13000.0,...,Never had,Never had,Yes,NaN,NaN,NaN,Yes,NaN,NaN,Low
ID_9OP2C8,malawi,28.0,Yes,No,No,No,No,180000.0,60000.0,30000.0,...,Never had,Never had,No,NaN,NaN,NaN,Yes,Never had,Have now,Low
ID_13REYS,zimbabwe,43.0,Yes,No,No,Yes,Yes,50.0,2400.0,1800.0,...,NaN,NaN,No,0.0,Never had,Never had,Yes,NaN,NaN,Low


In [6]:
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
owner_age,9618.0,4.170534e+01,1.331401e+01,18.0,32.0,40.0,50.0,103.0
personal_income,9509.0,2.627345e+05,2.566268e+06,0.0,300.0,2000.0,25000.0,150000000.0
business_expenses,9389.0,4.583838e+05,6.184746e+06,0.0,700.0,3000.0,25000.0,500000000.0
business_turnover,9402.0,1.348210e+06,8.804741e+06,0.0,1500.0,6000.0,50000.0,420000000.0
business_age_years,9366.0,7.030536e+00,7.650349e+00,0.0,2.0,4.0,10.0,60.0
business_age_months,5507.0,3.636281e+00,3.386488e+00,0.0,0.0,3.0,6.0,11.0


In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9618 entries, ID_3CFL0U to ID_4XGOHM
Data columns (total 38 columns):
 #   Column                                                            Non-Null Count  Dtype  
---  ------                                                            --------------  -----  
 0   country                                                           9618 non-null   object 
 1   owner_age                                                         9618 non-null   float64
 2   attitude_stable_business_environment                              9616 non-null   object 
 3   attitude_worried_shutdown                                         9616 non-null   object 
 4   compliance_income_tax                                             9614 non-null   object 
 5   perception_insurance_doesnt_cover_losses                          9613 non-null   object 
 6   perception_cannot_afford_insurance                                9613 non-null   object 
 7   personal_income          

In [8]:
missing = pd.DataFrame({
    'count': train_df.isnull().sum(),
    'pct': (train_df.isnull().sum() / len(train_df) * 100).round(2)
})
print(missing[missing['count'] > 0])

                                                                  count    pct
attitude_stable_business_environment                                  2   0.02
attitude_worried_shutdown                                             2   0.02
compliance_income_tax                                                 4   0.04
perception_insurance_doesnt_cover_losses                              5   0.05
perception_cannot_afford_insurance                                    5   0.05
personal_income                                                     109   1.13
business_expenses                                                   229   2.38
business_turnover                                                   216   2.25
business_age_years                                                  252   2.62
motor_vehicle_insurance                                            2244  23.33
has_mobile_money                                                   2751  28.60
current_problem_cash_flow                           

In [9]:
missing = pd.DataFrame({
    'count': test_df.isnull().sum(),
    'pct': (test_df.isnull().sum() / len(train_df) * 100).round(2)
})
print(missing[missing['count'] > 0])

                                                                  count    pct
owner_age                                                             1   0.01
perception_insurance_doesnt_cover_losses                              2   0.02
perception_cannot_afford_insurance                                    2   0.02
personal_income                                                      23   0.24
business_expenses                                                    70   0.73
business_turnover                                                    70   0.73
business_age_years                                                   59   0.61
motor_vehicle_insurance                                             557   5.79
has_mobile_money                                                    710   7.38
current_problem_cash_flow                                           932   9.69
has_cellphone                                                       486   5.05
owner_sex                                           

In [10]:
test_df

,country,owner_age,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,personal_income,business_expenses,business_turnover,...,has_loan_account,has_internet_banking,has_debit_card,future_risk_theft_stock,business_age_months,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender
ID,,,,,,,,,,,,,,,,,,,,,
ID_5EGLKX,zimbabwe,50.0,No,No,No,No,Yes,100.0,3600.0,7200.0,...,NaN,NaN,NaN,No,8.0,Never had,Never had,NaN,NaN,NaN
ID_4AI7RE,lesotho,36.0,Yes,Yes,No,Yes,Yes,900.0,400.0,900.0,...,NaN,NaN,NaN,Yes,NaN,NaN,NaN,Yes,Used to have but don't have now,Used to have but don't have now
ID_V9OB3M,lesotho,25.0,Don’t know or N/A,No,No,Don't know,Don't know,5250.0,350.0,1000.0,...,Used to have but don't have now,Have now,Have now,Yes,NaN,NaN,NaN,No,Never had,Used to have but don't have now
ID_6OI9DI,malawi,25.0,Don’t know or N/A,Yes,No,No,Yes,485000.0,10000.0,20000.0,...,Never had,Never had,Never had,Yes,NaN,NaN,NaN,Yes,Have now,Never had
ID_H2TN8B,lesotho,47.0,No,Yes,No,Don't know,Don't know,97.0,500.0,2000.0,...,Used to have but don't have now,Have now,Have now,Yes,NaN,NaN,NaN,Yes,Used to have but don't have now,Used to have but don't have now
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ID_FX7XJZ,eswatini,29.0,Yes,Yes,No,No,Yes,600.0,1700.0,2000.0,...,Never had,Never had,Never had,NaN,11.0,Never had,Never had,NaN,Never had,Never had
ID_XAL1LX,malawi,20.0,Don’t know or N/A,Don’t know or N/A,No,Don't know,Don't know,30000.0,20000.0,25000.0,...,Never had,Never had,Never had,No,4.0,NaN,NaN,Yes,NaN,NaN
ID_UHBP0F,zimbabwe,26.0,Yes,Yes,No,Yes,Yes,3888.0,NaN,NaN,...,NaN,NaN,NaN,No,0.0,Have now,Have now,NaN,NaN,NaN


In [11]:
# Number of unique categories
train_df['Target'].nunique()

# Count per category
train_df['Target'].value_counts()

Target
Low       6280
Medium    2868
High       470
Name: count, dtype: int64

## Prepare data for Machine Learning

In [12]:
#splits = RandomSplitter(valid_pct=0.2)(range_of(original_df))
#train_df = pd.concat([train_df, original_df], ignore_index=True) *

cont_names,cat_names = cont_cat_split(train_df, dep_var='Target')

In [13]:
splits = RandomSplitter(valid_pct=0.2)(range_of(train_df))

In [14]:
to = TabularPandas(train_df, procs=[Categorify, FillMissing,Normalize],
                   cat_names = cat_names,
                   cont_names = cont_names,
                   y_names='Target',
                   y_block=CategoryBlock(),
                   splits=splits)

/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  to[n].fillna(self.na_dict[n], inplace=True)
/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as

In [15]:
cont_names

['owner_age',
 'personal_income',
 'business_expenses',
 'business_turnover',
 'business_age_years',
 'business_age_months']

In [16]:
dls = to.dataloaders(bs=64)

In [17]:
dls.show_batch()

,country,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,motor_vehicle_insurance,has_mobile_money,current_problem_cash_flow,has_cellphone,owner_sex,offers_credit_to_customers,attitude_satisfied_with_achievement,has_credit_card,keeps_financial_records,perception_insurance_companies_dont_insure_businesses_like_yours,perception_insurance_important,has_insurance,covid_essential_service,attitude_more_successful_next_year,problem_sourcing_money,marketing_word_of_mouth,has_loan_account,has_internet_banking,has_debit_card,future_risk_theft_stock,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,personal_income_na,business_expenses_na,business_turnover_na,business_age_years_na,business_age_months_na,owner_age,personal_income,business_expenses,business_turnover,business_age_years,business_age_months,Target
0,zimbabwe,Yes,Yes,No,Don't know,Don't know,Never had,#na#,Yes,No,Female,No,Yes,Never had,"Yes, sometimes",Don't know,Do not know / N‎/A,No,No,#na#,#na#,#na#,#na#,#na#,#na#,Yes,Never had,Never had,Yes,#na#,#na#,False,False,False,False,False,24.999999,3.699924e+02,2400.000685,3.599950e+03,6.0,1.416142e-07,Low
1,malawi,Don’t know or N/A,Yes,No,Don't know,Yes,#na#,Never had,No,No,Female,"Yes, always",Yes,Never had,No,Yes,No,No,#na#,Yes,Yes,No,Never had,Never had,Never had,No,#na#,#na#,No,#na#,#na#,False,False,False,False,True,42.000000,1.080000e+06,539999.998932,1.080000e+06,11.0,3.000000e+00,Low
2,malawi,Don’t know or N/A,Yes,No,Yes,Yes,#na#,Have now,Yes,Yes,Male,"Yes, sometimes",No,Never had,Yes,Don?t know / doesn?t apply,Yes,No,#na#,Don't know,Yes,No,Never had,Never had,Never had,No,#na#,#na#,Yes,#na#,#na#,False,False,False,False,True,26.000000,5.000000e+04,30000.007455,2.999994e+04,5.0,3.000000e+00,Low
3,lesotho,Yes,No,No,No,Yes,Never had,Have now,Yes,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,No,Yes,No,Yes,Used to have but don't have now,Have now,Have now,#na#,#na#,#na#,No,Used to have but don't have now,Used to have but don't have now,False,False,False,False,True,28.000000,1.999999e+03,2999.984538,5.000037e+03,10.0,3.000000e+00,Low
4,lesotho,Yes,Yes,No,Yes,Yes,Never had,Have now,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,Don't know,Yes,No,Yes,Used to have but don't have now,Never had,Never had,#na#,#na#,#na#,Yes,Used to have but don't have now,Used to have but don't have now,False,False,False,False,True,44.000000,7.000003e+03,199.993266,9.999794e+02,1.0,3.000000e+00,Low
5,malawi,No,No,No,Don't know,Don't know,#na#,Have now,No,Yes,Female,"Yes, sometimes",No,Never had,No,Don?t know / doesn?t apply,Don?t know / doesn?t apply,No,#na#,Yes,No,No,Never had,Never had,Never had,No,#na#,#na#,Yes,#na#,#na#,False,False,False,True,False,32.000000,5.200000e+05,45000.003530,6.260000e+06,4.0,8.000000e+00,Low
6,eswatini,No,No,No,No,Yes,Never had,Have now,Yes,Yes,Male,"Yes, always",No,Never had,No,No,Yes,No,No,Yes,Yes,Yes,Never had,Never had,Never had,#na#,Never had,Used to have but don’t have now,#na#,Never had,Never had,False,False,False,False,False,58.000001,1.000006e+03,1300.021960,2.499986e+03,2.0,6.000000e+00,Low
7,lesotho,Yes,No,No,Don't know,Yes,Never had,Have now,Yes,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,Don't know,Yes,Yes,Yes,#na#,#na#,#na#,#na#,#na#,#na#,No,Used to have but don't have now,Used to have but don't have now,False,False,False,False,True,39.000000,4.999992e+02,100.004285,5.000177e+02,2.0,3.000000e+00,Medium
8,malawi,Don’t know or N/A,No,Yes,No,Yes,#na#,Never had,Yes,Yes,Female,No,No,Have now,Yes,No,Don?t know / doesn?t apply,No,#na#,Yes,No,Yes,Never had,Never had,Have now,No,#na#,#na#,No,#na#,#na#,False,False,False,False,True,36.000000,1.455000e+04,312000.001126,3.145500e+05,4.0,3.000000e+00,Medium
9,zimbabwe,Yes,Don’t know or N/A,No,No,Yes,Never had,Have now,Yes,Yes,Male,No,Yes,Never had,No,No,Yes,No,No,#na#,#na#,#na#,#na#,#na#,#na#,No,Nev

In [18]:
learn = tabular_learner(dls, metrics=F1ScoreMulti(average='macro'))

In [19]:
learn = tabular_learner(dls, metrics=F1Score(average='macro'))

In [20]:
learn.fit_one_cycle(50)

epoch,train_loss,valid_loss,f1_score,time
0,1.085382,1.074330,0.383792,00:01
1,1.019640,0.966779,0.453516,00:01
2,0.815484,0.712001,0.530218,00:01
3,0.571171,0.606600,0.659943,00:01
4,0.459812,0.483293,0.757276,00:01
5,0.399505,0.471081,0.757039,00:01
6,0.373385,0.411680,0.799205,00:01
7,0.347767,0.371722,0.806084,00:01
8,0.319367,0.429453,0.783650,00:01
9,0.302277,0.401387,0.804695,00:01


In [21]:
learn.show_results()

,country,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,motor_vehicle_insurance,has_mobile_money,current_problem_cash_flow,has_cellphone,owner_sex,offers_credit_to_customers,attitude_satisfied_with_achievement,has_credit_card,keeps_financial_records,perception_insurance_companies_dont_insure_businesses_like_yours,perception_insurance_important,has_insurance,covid_essential_service,attitude_more_successful_next_year,problem_sourcing_money,marketing_word_of_mouth,has_loan_account,has_internet_banking,has_debit_card,future_risk_theft_stock,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,personal_income_na,business_expenses_na,business_turnover_na,business_age_years_na,business_age_months_na,owner_age,personal_income,business_expenses,business_turnover,business_age_years,business_age_months,Target,Target_pred
0,1.0,3.0,2.0,4.0,2.0,3.0,3.0,2.0,1.0,2.0,1.0,3.0,4.0,3.0,4.0,4.0,5.0,1.0,2.0,4.0,2.0,2.0,4.0,4.0,3.0,0.0,4.0,2.0,0.0,4.0,4.0,1.0,1.0,1.0,1.0,1.0,1.834758,-0.097535,-0.067449,-0.160656,0.812915,0.636514,2.0,2.0
1,3.0,3.0,2.0,2.0,2.0,3.0,0.0,2.0,3.0,1.0,1.0,3.0,3.0,3.0,2.0,4.0,5.0,1.0,0.0,4.0,1.0,1.0,4.0,4.0,3.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,2.0,-0.426367,0.646925,1.870526,1.753525,-0.788224,-0.137258,1.0,1.0
2,1.0,3.0,2.0,4.0,2.0,3.0,3.0,2.0,1.0,2.0,2.0,3.0,4.0,3.0,3.0,4.0,5.0,1.0,3.0,4.0,2.0,2.0,4.0,3.0,2.0,0.0,4.0,3.0,0.0,4.0,6.0,1.0,1.0,1.0,1.0,1.0,-1.029333,-0.095486,-0.065869,-0.158692,-0.654796,2.184057,1.0,1.0
3,3.0,2.0,3.0,2.0,3.0,3.0,0.0,2.0,2.0,2.0,2.0,1.0,4.0,3.0,2.0,4.0,4.0,1.0,0.0,3.0,1.0,2.0,4.0,4.0,4.0,1.0,0.0,0.0,2.0,0.0,0.0,1.0,1.0,1.0,1.0,2.0,0.779566,0.721427,-0.064825,0.510829,3.348052,-0.137258,1.0,1.0
4,1.0,2.0,3.0,2.0,2.0,3.0,3.0,2.0,1.0,2.0,2.0,3.0,3.0,3.0,3.0,4.0,4.0,1.0,3.0,4.0,2.0,0.0,4.0,4.0,3.0,0.0,4.0,3.0,0.0,4.0,4.0,1.0,1.0,1.0,1.0,1.0,-0.577108,-0.097878,-0.065123,-0.158201,0.012346,2.957829,1.0,1.0
5,1.0,3.0,2.0,2.0,2.0,3.0,3.0,2.0,3.0,2.0,2.0,3.0,3.0,3.0,3.0,4.0,5.0,1.0,2.0,4.0,2.0,2.0,6.0,4.0,3.0,0.0,4.0,3.0,0.0,4.0,4.0,1.0,1.0,1.0,1.0,1.0,1.382533,-0.097051,-0.067539,-0.154372,-0.921652,1.797171,2.0,1.0
6,3.0,1.0,3.0,2.0,1.0,1.0,0.0,3.0,3.0,2.0,1.0,3.0,3.0,3.0,2.0,3.0,5.0,1.0,0.0,4.0,1.0,1.0,4.0,4.0,2.0,1.0,0.0,0.0,2.0,3.0,4.0,1.0,1.0,1.0,1.0,2.0,-0.426367,-0.079468,-0.066018,-0.158692,0.012346,-0.137258,1.0,1.0
7,3.0,1.0,3.0,4.0,1.0,1.0,0.0,2.0,2.0,1.0,1.0,2.0,3.0,3.0,2.0,3.0,5.0,1.0,0.0,4.0,1.0,1.0,4.0,4.0,3.0,2.0,0.0,0.0,2.0,0.0,0.0,1.0,1.0,1.0,1.0,2.0,-1.481558,-0.004967,-0.045442,0.148146,-0.788224,-0.137258,1.0,1.0
8,4.0,3.0,3.0,2.0,1.0,3.0,3.0,0.0,3.0,1.0,1.0,1.0,3.0,3.0,3.0,2.0,5.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,3.0,2.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,-0.803221,-0.098072,-0.067753,-0.161059,-0.788224,-1.297915,1.0,1.0


In [22]:
test_to = to.new(test_df)

In [23]:
test_processed = test_to.items.copy()
test_processed.shape, test_df.shape

((2405, 37), (2405, 37))

In [24]:
test_processed['owner_age'].fillna(test_processed['owner_age'].median(), inplace=True)

/tmp/ipykernel_66673/2900860375.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_processed['owner_age'].fillna(test_processed['owner_age'].median(), inplace=True)


In [25]:
test_dl = dls.test_dl(test_processed)
test_dl

/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  to[n].fillna(self.na_dict[n], inplace=True)
/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as

In [26]:
#dl = learn.dls.test_dl(test_dl)

In [27]:
dl = learn.dls.test_dl(test_processed)

/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  to[n].fillna(self.na_dict[n], inplace=True)
/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as

In [28]:
learn.get_preds(dl=dl)

(tensor([[1.4126e-04, 9.7848e-01, 2.1375e-02],
         [2.6323e-04, 6.9684e-01, 3.0290e-01],
         [2.9177e-06, 9.9996e-01, 4.1805e-05],
         ...,
         [2.2110e-07, 6.8395e-06, 9.9999e-01],
         [4.5709e-03, 2.9062e-05, 9.9540e-01],
         [2.6004e-06, 9.9999e-01, 1.1975e-05]]),
 None)

In [29]:
xc = learn.get_preds(dl=dl)
xc

(tensor([[1.4126e-04, 9.7848e-01, 2.1375e-02],
         [2.6323e-04, 6.9684e-01, 3.0290e-01],
         [2.9177e-06, 9.9996e-01, 4.1805e-05],
         ...,
         [2.2110e-07, 6.8395e-06, 9.9999e-01],
         [4.5709e-03, 2.9062e-05, 9.9540e-01],
         [2.6004e-06, 9.9999e-01, 1.1975e-05]]),
 None)

## Creating Submission File from FastAI Predictions

### What does `learn.get_preds(dl=dl)` do?

`learn.get_preds(dl=dl)` is a FastAI method that generates predictions from your trained neural network model:

**Returns a tuple:**
- **First element**: A tensor of predicted probabilities with shape `(n_samples, n_classes)`
  - In our case: `(2405, 3)` - 2405 test samples × 3 classes (High, Low, Medium)
  - Each row contains 3 probability values that sum to ~1.0
  - Example: `[0.00000171, 0.98901, 0.010984]` means:
    - 0.0002% probability of "High"
    - 98.9% probability of "Low" 
    - 1.1% probability of "Medium"
- **Second element**: `None` (would contain targets if they existed in the test set)

**How it works:**
1. Takes your test dataloader (`dl`) containing preprocessed test data
2. Passes each sample through your trained neural network
3. Applies softmax to get probability distributions for each class
4. Returns the probabilities for all test samples

### Creating a Submission File

To create a submission file, follow these steps:

1. **Get predictions**: Call `learn.get_preds(dl=dl)` to get probability tensor
2. **Find winning class**: Use `argmax()` to get the index of highest probability
3. **Map to labels**: Convert numeric indices (0, 1, 2) to class names using `to.vocab`
4. **Create DataFrame**: Build submission with ID and Target columns
5. **Save to CSV**: Export in the format expected by the competition

The code in the next cell demonstrates this complete workflow.

In [30]:
# Create submission file from FastAI neural network predictions

# Step 1: Get predictions (probabilities for each class)
preds, _ = learn.get_preds(dl=dl)

# Step 2: Convert probabilities to class indices (0, 1, 2)
# argmax gets the index of the highest probability
pred_classes = preds.argmax(dim=1)

# Step 3: Convert numeric indices to actual class names (High, Low, Medium)
# to.vocab contains the mapping from indices to class labels
pred_labels = [to.vocab[i] for i in pred_classes]

# Step 4: Create submission dataframe matching the sample submission format
submission = pd.DataFrame({
    'ID': test_df.index,  # IDs from test set
    'Target': pred_labels
})

# Step 5: Verify format matches sample submission
print(f"Submission shape: {submission.shape}")
print(f"Sample submission shape: {sub_df.shape}")
print(f"\nClass distribution in predictions:")
print(submission['Target'].value_counts())
print(f"\nFirst few predictions:")
print(submission.head())

# Step 6: Save to CSV
submission.to_csv('submission_fastai.csv', index=False)
print(f"\nSubmission saved to submission_fastai.csv")

Submission shape: (2405, 2)
Sample submission shape: (2405, 2)

Class distribution in predictions:
Target
Low       1586
Medium     717
High       102
Name: count, dtype: int64

First few predictions:
          ID Target
0  ID_5EGLKX    Low
1  ID_4AI7RE    Low
2  ID_V9OB3M    Low
3  ID_6OI9DI    Low
4  ID_H2TN8B    Low

Submission saved to submission_fastai.csv


In [31]:
len(xc)

2

In [32]:
xc[0].shape

torch.Size([2405, 3])

In [33]:
xcy = xc[0]
xcy

tensor([[1.4126e-04, 9.7848e-01, 2.1375e-02],
        [2.6323e-04, 6.9684e-01, 3.0290e-01],
        [2.9177e-06, 9.9996e-01, 4.1805e-05],
        ...,
        [2.2110e-07, 6.8395e-06, 9.9999e-01],
        [4.5709e-03, 2.9062e-05, 9.9540e-01],
        [2.6004e-06, 9.9999e-01, 1.1975e-05]])

In [76]:
sub_df

,ID,Target
0,ID_5EGLKX,Low
1,ID_4AI7RE,Low
2,ID_V9OB3M,Low
3,ID_6OI9DI,Low
4,ID_H2TN8B,Low
...,...,...
2400,ID_FX7XJZ,Low
2401,ID_XAL1LX,Low
2402,ID_UHBP0F,Medium
2403,ID_GKIKR2,Medium


In [35]:
X_train, y_train = to.train.xs, to.train.ys.values.ravel()
X_test, y_test = to.valid.xs, to.valid.ys.values.ravel()

In [36]:
X_train

,country,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,motor_vehicle_insurance,has_mobile_money,current_problem_cash_flow,has_cellphone,...,business_expenses_na,business_turnover_na,business_age_years_na,business_age_months_na,owner_age,personal_income,business_expenses,business_turnover,business_age_years,business_age_months
ID,,,,,,,,,,,,,,,,,,,,,
ID_J2XORV,1,3,3,4,2,3,3,3,1,1,...,1,1,1,1,-0.953963,-0.097349,-0.062589,-0.143964,1.213200,0.636514
ID_GOCRK5,2,3,1,1,3,3,3,0,3,0,...,1,1,1,2,-0.878592,-0.097535,-0.067509,-0.160533,-0.521367,-0.137258
ID_0BHNL9,4,3,2,2,2,3,3,0,0,2,...,1,1,1,1,0.251971,-0.098004,-0.067607,-0.161032,-0.788224,-1.297915
ID_CIFL79,2,3,2,2,2,2,1,2,3,0,...,1,1,1,2,-0.124884,-0.097945,-0.067750,-0.161098,1.346628,-0.137258
ID_6COXKQ,2,1,1,2,1,3,3,0,3,0,...,1,1,1,2,1.081050,-0.098058,-0.067793,-0.161135,3.081196,-0.137258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ID_JV9LHA,1,2,2,4,3,3,3,3,3,2,...,1,1,1,1,2.814578,-0.060843,-0.056028,-0.148874,4.415479,-1.297915
ID_9JP50E,3,2,2,2,1,1,0,2,2,2,...,1,1,1,2,0.553454,-0.004967,-0.049915,-0.137827,-0.254511,-0.137258
ID_J4BIOH,2,2,3,4,2,2,3,2,0,0,...,1,1,1,2,-0.953963,-0.086919,-0.065571,-0.158692,0.812915,-0.137258


## Modelling

In [47]:
# ==========================================
# XGBoost Default Parameters (Baseline)
# ==========================================
# These are the default values for XGBoost multiclass classification
# Use this as your baseline before hyperparameter tuning

xgb_default_params = {
    # ==========================================
    # TASK PARAMETERS (Required for multiclass)
    # ==========================================
    'objective': 'multi:softprob',      # Multiclass with probability output
    'num_class': 3,                     # 3 classes: Low, Medium, High
    'eval_metric': 'mlogloss',          # Multiclass log loss
    
    # ==========================================
    # TREE STRUCTURE (Core parameters)
    # ==========================================
    'max_depth': 6,                     # DEFAULT: 6 - Maximum tree depth
    'min_child_weight': 1,              # DEFAULT: 1 - Min sum of instance weight in child
    'gamma': 0,                         # DEFAULT: 0 - Min loss reduction for split
    
    # ==========================================
    # LEARNING RATE & BOOSTING
    # ==========================================
    'learning_rate': 0.3,               # DEFAULT: 0.3 (eta) - Step size shrinkage
    'n_estimators': 100,                # DEFAULT: 100 - Number of boosting rounds
    
    # ==========================================
    # SAMPLING (Prevents overfitting)
    # ==========================================
    'subsample': 1.0,                   # DEFAULT: 1.0 - Row sampling ratio
    'colsample_bytree': 1.0,            # DEFAULT: 1.0 - Column sampling per tree
    'colsample_bylevel': 1.0,           # DEFAULT: 1.0 - Column sampling per level
    'colsample_bynode': 1.0,            # DEFAULT: 1.0 - Column sampling per node
    
    # ==========================================
    # REGULARIZATION
    # ==========================================
    'reg_alpha': 0,                     # DEFAULT: 0 - L1 regularization (Lasso)
    'reg_lambda': 1,                    # DEFAULT: 1 - L2 regularization (Ridge)
    
    # ==========================================
    # CLASS IMBALANCE (Your case: High=4.9%)
    # ==========================================
    'max_delta_step': 0,                # DEFAULT: 0 - Max output change (0=no limit)
    # Note: For multiclass imbalance, use sample_weight in fit() instead of scale_pos_weight
    
    # ==========================================
    # SYSTEM PARAMETERS
    # ==========================================
    'tree_method': 'hist',              # Use 'hist' for speed (default: 'auto')
    'random_state': 42,                 # For reproducibility (default: 0)
    'n_jobs': -1,                       # Use all CPU cores (default: None)
    'verbosity': 0,                     # Silent mode (default: 1)
}

# Print summary
print("="*70)
print("XGBoost Default Parameters Loaded")
print("="*70)
print(f"Total parameters: {len(xgb_default_params)}")
print(f"\nKey settings:")
print(f"  Objective: {xgb_default_params['objective']}")
print(f"  Classes: {xgb_default_params['num_class']}")
print(f"  Max depth: {xgb_default_params['max_depth']}")
print(f"  Learning rate: {xgb_default_params['learning_rate']}")
print(f"  N estimators: {xgb_default_params['n_estimators']}")
print(f"  Tree method: {xgb_default_params['tree_method']}")
print("="*70)

# Train with default parameters using 5-Fold Stratified CV
print("\n" + "="*70)
print("Training XGBoost with DEFAULT parameters using 5-Fold Stratified CV...")
print("="*70)

# Setup cross-validation
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores_default = []

# Perform cross-validation
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\nFold {fold + 1}/5")
    
    # Split data for this fold
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Create and train model
    model_fold = xgb.XGBClassifier(**xgb_default_params)
    model_fold.fit(X_tr, y_tr, verbose=False)
    
    # Predict and evaluate
    y_pred = model_fold.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    f1_scores_default.append(f1)
    
    print(f"  Fold {fold + 1} F1 Macro: {f1:.6f}")

# Calculate mean and std
mean_f1_default = np.mean(f1_scores_default)
std_f1_default = np.std(f1_scores_default)

# Show results
print("\n" + "="*70)
print("RESULTS: Default Parameters with 5-Fold Cross-Validation")
print("="*70)
print(f"Mean F1 Macro Score: {mean_f1_default:.6f} (+/- {std_f1_default:.6f})")
print(f"\nFold scores: {[f'{score:.6f}' for score in f1_scores_default]}")
print(f"Min: {min(f1_scores_default):.6f}")
print(f"Max: {max(f1_scores_default):.6f}")
print("="*70)

# Train final model on full training data
print("\nTraining final model on full training data...")
model_default = xgb.XGBClassifier(**xgb_default_params)
model_default.fit(X_train, y_train, verbose=False)
print("✅ Final model trained!")
print("="*70)

XGBoost Default Parameters Loaded
Total parameters: 19

Key settings:
  Objective: multi:softprob
  Classes: 3
  Max depth: 6
  Learning rate: 0.3
  N estimators: 100
  Tree method: hist


In [55]:
# ==========================================
# XGBoost BETTER Starting Parameters
# ==========================================
# These are IMPROVED values (NOT defaults) for better baseline performance
# Expected improvement: F1 ~ 0.810-0.815 (vs default 0.8002)

xgb_better_params = {
    # ==========================================
    # TASK PARAMETERS (Same as defaults)
    # ==========================================
    'device': 'cuda',
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    
    # ==========================================
    # TREE STRUCTURE (Keep conservative defaults)
    # ==========================================
    'max_depth': 6,                     # SAME - Good balance
    'min_child_weight': 1,              # SAME - Allow flexibility
    'gamma': 0,                         # SAME - No penalty initially
    
    # ==========================================
    # LEARNING RATE & BOOSTING (IMPROVED)
    # ==========================================
    'learning_rate': 0.1,               # CHANGED from 0.3 → Better generalization
    'n_estimators': 500,                # CHANGED from 100 → More learning capacity
    
    # ==========================================
    # SAMPLING (IMPROVED - Prevents overfitting)
    # ==========================================
    'subsample': 0.8,                   # CHANGED from 1.0 → Add row sampling
    'colsample_bytree': 0.8,            # CHANGED from 1.0 → Add column sampling
    'colsample_bylevel': 1.0,           # SAME - Not needed yet
    'colsample_bynode': 1.0,            # SAME - Not needed yet
    
    # ==========================================
    # REGULARIZATION (Keep defaults for now)
    # ==========================================
    'reg_alpha': 0,                     # SAME - No L1 yet
    'reg_lambda': 1,                    # SAME - Default L2
    
    # ==========================================
    # CLASS IMBALANCE (Same as defaults)
    # ==========================================
    'max_delta_step': 0,                # SAME - No capping yet
    
    # ==========================================
    # SYSTEM PARAMETERS
    # ==========================================
    'tree_method': 'hist',
    'predictor': 'gpu_predictor',
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0,
}

# Create Stratified K-Fold for cross-validation
from sklearn.model_selection import StratifiedKFold

print("Training XGBoost with BETTER parameters using 5-Fold Stratified CV...")
print("="*70)

# Setup cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores_better = []

# Perform cross-validation
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\nFold {fold + 1}/5")
    
    # Split data for this fold
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Create and train model
    model_fold = xgb.XGBClassifier(**xgb_better_params)
    model_fold.fit(X_tr, y_tr, verbose=False)
    
    # Predict and evaluate
    y_pred = model_fold.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    f1_scores_better.append(f1)
    
    print(f"  Fold {fold + 1} F1 Macro: {f1:.6f}")

# Calculate mean and std
mean_f1_better = np.mean(f1_scores_better)
std_f1_better = np.std(f1_scores_better)

# Show results
print("\n" + "="*70)
print("RESULTS: Better Parameters with 5-Fold Cross-Validation")
print("="*70)
print(f"Mean F1 Macro Score: {mean_f1_better:.6f} (+/- {std_f1_better:.6f})")
print(f"\nFold scores: {[f'{score:.6f}' for score in f1_scores_better]}")
print(f"Min: {min(f1_scores_better):.6f}")
print(f"Max: {max(f1_scores_better):.6f}")
print("="*70)

# Train final model on full training data for later use
print("\nTraining final model on full training data...")
model_better = xgb.XGBClassifier(**xgb_better_params)
model_better.fit(X_train, y_train, verbose=False)
print("✅ Final model trained!")
print("="*70)

# Show what changed from defaults
print("\nKey changes from defaults:")
print(f"  learning_rate:     0.3 → {xgb_better_params['learning_rate']}")
print(f"  n_estimators:      100 → {xgb_better_params['n_estimators']}")
print(f"  subsample:         1.0 → {xgb_better_params['subsample']}")
print(f"  colsample_bytree:  1.0 → {xgb_better_params['colsample_bytree']}")
print("="*70)

Training XGBoost with BETTER parameters (no sample weights)...
RESULTS: Better Starting Parameters
Validation F1 Macro Score: 0.812445
Baseline (default params):  0.8002
Improvement:               +0.012245
Percentage improvement:    +1.53%

✅ Better parameters improved the score!

Key changes from defaults:
  learning_rate:     0.3 → 0.1
  n_estimators:      100 → 500
  subsample:         1.0 → 0.8
  colsample_bytree:  1.0 → 0.8


In [56]:
xgb_better_params

{'objective': 'multi:softprob',
 'num_class': 3,
 'eval_metric': 'mlogloss',
 'max_depth': 6,
 'min_child_weight': 1,
 'gamma': 0,
 'learning_rate': 0.1,
 'n_estimators': 500,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'colsample_bylevel': 1.0,
 'colsample_bynode': 1.0,
 'reg_alpha': 0,
 'reg_lambda': 1,
 'max_delta_step': 0,
 'tree_method': 'hist',
 'random_state': 42,
 'n_jobs': -1,
 'verbosity': 0}

In [57]:
%%time
xgb_clf = xgb.XGBClassifier(**xgb_better_params)
xgb_clf

CPU times: user 66 μs, sys: 2 μs, total: 68 μs
Wall time: 87.3 μs


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=1.0, colsample_bynode=1.0, colsample_bytree=0.8,
              device=None, early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, gamma=0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=0,
              max_depth=6, max_leaves=None, min_child_weight=1, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_class=3, num_parallel_tree=None, ...)

In [58]:
xgb_clf_fit = xgb_clf.fit(X_train, y_train)
xgb_clf_fit

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=1.0, colsample_bynode=1.0, colsample_bytree=0.8,
              device=None, early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, gamma=0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=0,
              max_depth=6, max_leaves=None, min_child_weight=1, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_class=3, num_parallel_tree=None, ...)

In [59]:
#xgb_preds = xgb_clf_fit.predict(test_dl.xs)
xgb_sc_preds = xgb_clf_fit.predict(X_test)

In [40]:
xgb_sc = f1_score(y_test, xgb_sc_preds, average='macro')
xgb_sc

0.8002434487899772

In [53]:
#new trial with default params score
xgb_sc = f1_score(y_test, xgb_sc_preds, average='macro')
xgb_sc

0.8002434487899772

In [60]:
#new trial with xgb better params score
xgb_sc = f1_score(y_test, xgb_sc_preds, average='macro')
xgb_sc

0.8124452175684628

In [61]:
xgb_preds = xgb_clf_fit.predict(test_dl.xs)

In [62]:
xgb_preds.shape, test_df.shape

((2405,), (2405, 37))

In [63]:
sub_df

,ID,Target
0,ID_5EGLKX,Low
1,ID_4AI7RE,Low
2,ID_V9OB3M,Low
3,ID_6OI9DI,Low
4,ID_H2TN8B,Low
...,...,...
2400,ID_FX7XJZ,Low
2401,ID_XAL1LX,Low
2402,ID_UHBP0F,Medium
2403,ID_GKIKR2,Medium


In [66]:
!rm submission.csv
xgb_names = to.vocab[xgb_preds]
submit = sub_df
submit.Target = xgb_names
submit.to_csv('submission.csv',index=False)
submit

,ID,Target
0,ID_5EGLKX,Low
1,ID_4AI7RE,Low
2,ID_V9OB3M,Low
3,ID_6OI9DI,Low
4,ID_H2TN8B,Low
...,...,...
2400,ID_FX7XJZ,Low
2401,ID_XAL1LX,Low
2402,ID_UHBP0F,Medium
2403,ID_GKIKR2,Medium
